# 02 Baselines

**Goal:** build the train, validation and test split from the EDA, set a rule based baseline, and fit a first logistic regression.

Decisions carried over from `01_eda.ipynb`:
- Split inside 1 to 10 Sep 2022, with the 11 to 18 Sep tail held apart.
- `self_transfer` is a feature. Payment format and payment currency are ablated.
- Metrics are PR-AUC and recall at a fixed false positive rate.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")

TX_COLS = [
    "timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid",
    "payment_currency", "payment_format", "is_laundering",
]
TX_DTYPES = {
    "from_bank": "int32", "from_account": "category",
    "to_bank": "int32", "to_account": "category",
    "amount_received": "float64", "receiving_currency": "category",
    "amount_paid": "float64", "payment_currency": "category",
    "payment_format": "category", "is_laundering": "int8",
}


def load_transactions(path):
    tx = pd.read_csv(path, header=0, names=TX_COLS, dtype=TX_DTYPES)
    tx["timestamp"] = pd.to_datetime(tx["timestamp"], format="%Y/%m/%d %H:%M")
    tx["date"] = tx["timestamp"].dt.normalize()
    tx["self_transfer"] = (
        (tx["from_account"].astype(str) == tx["to_account"].astype(str))
        & (tx["from_bank"] == tx["to_bank"])
    )
    return tx


tx = load_transactions(DATA / "HI-Small_Trans.csv")
print(f"rows: {len(tx):,}")

rows: 5,078,345


In [2]:
SPLITS = {
    "train": ("2022-09-01", "2022-09-06"),
    "validation": ("2022-09-07", "2022-09-08"),
    "test": ("2022-09-09", "2022-09-10"),
}


def split_by_date(tx, splits=SPLITS):
    return {
        name: tx[(tx["date"] >= start) & (tx["date"] <= end)]
        for name, (start, end) in splits.items()
    }


parts = split_by_date(tx)
train, val, test = parts["train"], parts["validation"], parts["test"]

for name, part in parts.items():
    print(f"{name:<11} rows: {len(part):>9,}   laundering: {part['is_laundering'].sum():>5,}")

train       rows: 3,248,921   laundering: 2,530
validation  rows:   965,524   laundering: 1,036
test        rows:   862,792   laundering:   956


## Evaluation

Every model is scored the same way on the validation set:
- **PR-AUC** (average precision), the main metric at this class balance.
- **Recall at a fixed false positive rate** of 0.1% and 1%, which answers "how much laundering is caught if analysts can only review this many alerts".

A model that ranks at random has a PR-AUC equal to the laundering rate, about 0.001.

In [3]:
from sklearn.metrics import average_precision_score, roc_curve


def recall_at_fpr(y_true, scores, max_fpr):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return float(tpr[fpr <= max_fpr].max())


def evaluate(y_true, scores, name, fprs=(0.001, 0.01)):
    result = {"model": name, "pr_auc": average_precision_score(y_true, scores)}
    for f in fprs:
        result[f"recall_at_{f:.1%}_fpr"] = recall_at_fpr(y_true, scores, f)
    return result


results = []
y_val = val["is_laundering"].to_numpy()
print(f"validation laundering rate: {y_val.mean():.4%}")

validation laundering rate: 0.1073%


In [4]:
ach_flag = (val["payment_format"] == "ACH").to_numpy().astype(int)

tp = int(((ach_flag == 1) & (y_val == 1)).sum())
fp = int(((ach_flag == 1) & (y_val == 0)).sum())
fn = int(((ach_flag == 0) & (y_val == 1)).sum())
tn = int(((ach_flag == 0) & (y_val == 0)).sum())

print(f"alerts raised: {tp + fp:,} of {len(val):,} transactions")
print(f"precision: {tp / (tp + fp):.2%}")
print(f"recall: {tp / (tp + fn):.2%}")
print(f"false positive rate: {fp / (fp + tn):.2%}")

results.append(evaluate(y_val, ach_flag, "ACH rule"))
pd.DataFrame(results)

alerts raised: 116,910 of 965,524 transactions
precision: 0.76%
recall: 85.91%
false positive rate: 12.03%


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,ACH rule,0.006691,0.0,0.0


## Features and logistic regression

First model: logistic regression on transaction level features only. It is fitted twice, without and with the two shortcut columns (payment format and payment currency), to see how much of the score depends on them.

In [5]:
currency_median = (
    train.groupby("payment_currency", observed=True)["amount_paid"].median().to_dict()
)


def add_features(df, currency_median):
    out = pd.DataFrame(index=df.index)
    out["log_amount"] = np.log10(df["amount_paid"].clip(lower=0.01))
    typical = df["payment_currency"].astype(str).map(currency_median)
    out["amount_vs_currency"] = out["log_amount"] - np.log10(typical)
    hour = df["timestamp"].dt.hour
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    out["self_transfer"] = df["self_transfer"].astype(int)
    out["payment_format"] = df["payment_format"].astype(str)
    out["payment_currency"] = df["payment_currency"].astype(str)
    return out


X_train = add_features(train, currency_median)
X_val = add_features(val, currency_median)
y_train = train["is_laundering"].to_numpy()

X_train.head()

,log_amount,amount_vs_currency,hour_sin,hour_cos,self_transfer,payment_format,payment_currency
0,3.567889,0.573775,0.0,1.0,1,Reinvestment,US Dollar
1,-2.000000,-4.994115,0.0,1.0,0,Cheque,US Dollar
2,4.166595,1.172480,0.0,1.0,1,Reinvestment,US Dollar
3,3.448238,0.454123,0.0,1.0,1,Reinvestment,US Dollar
4,4.564464,1.570350,0.0,1.0,1,Reinvestment,US Dollar


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["log_amount", "amount_vs_currency", "hour_sin", "hour_cos", "self_transfer"]
SHORTCUTS = ["payment_format", "payment_currency"]


def fit_logreg(X, y, categorical):
    steps = [("num", StandardScaler(), NUMERIC)]
    if categorical:
        steps.append(("cat", OneHotEncoder(handle_unknown="ignore"), categorical))
    model = make_pipeline(
        ColumnTransformer(steps),
        LogisticRegression(class_weight="balanced", max_iter=500),
    )
    return model.fit(X, y)


models = {}
for name, categorical in [
    ("LogReg, no shortcut columns", []),
    ("LogReg, with format and currency", SHORTCUTS),
]:
    models[name] = fit_logreg(X_train, y_train, categorical)
    scores = models[name].predict_proba(X_val)[:, 1]
    results.append(evaluate(y_val, scores, name))

pd.DataFrame(results)

,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,ACH rule,0.006691,0.000000,0.000000
1,"LogReg, no shortcut columns",0.002210,0.001931,0.018340
2,"LogReg, with format and currency",0.016349,0.031853,0.131274


**Finding:** Logistic regression on transaction level features alone reaches a PR-AUC of 0.0022, about twice a random ranking and below the ACH rule (0.0067). At a 1% false positive rate it catches 19 of 1,036 laundering transactions. Adding payment format and payment currency raises PR-AUC to 0.0163 and recall at a 1% false positive rate to 136 of 1,036, so almost all of the model's skill comes from the two shortcut columns. A single transaction viewed in isolation carries little signal, which fits the multi-hop structure seen in the patterns file. The linear model also cannot represent the amount band found in the EDA, where laundering is concentrated between roughly 1,000 and 20,000.

## Gradient boosting

Same features as the logistic regression, with a non-linear model. This tests whether the weak result without shortcut columns was a limit of the linear model or of the features.

LightGBM is trained with L2 regularisation and early stopping on validation PR-AUC. An earlier run without them, fixed at 300 trees, was unstable at this class balance: validation PR-AUC peaked near 100 trees and then fell, and single late trees produced extreme gain values.

In [7]:
import lightgbm as lgb

CATEGORIES = {c: sorted(X_train[c].unique()) for c in SHORTCUTS}


def gbm_frame(X, use_shortcuts, numeric=NUMERIC):
    cols = numeric + (SHORTCUTS if use_shortcuts else [])
    out = X[cols].copy()
    for c in out.columns:
        if c in CATEGORIES:
            out[c] = pd.Categorical(out[c], categories=CATEGORIES[c])
    return out


def fit_gbm(X_tr, y_tr, X_va, y_va, use_shortcuts, numeric=NUMERIC, seed=42):
    model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=200,
        reg_lambda=10.0,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        metric="average_precision",
        random_state=seed,
        verbose=-1,
    )
    model.fit(
        gbm_frame(X_tr, use_shortcuts, numeric),
        y_tr,
        eval_set=[(gbm_frame(X_va, use_shortcuts, numeric), y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    return model


def run_gbm(name, X_tr, X_va, use_shortcuts, numeric=NUMERIC):
    model = fit_gbm(X_tr, y_train, X_va, y_val, use_shortcuts, numeric)
    scores = model.predict_proba(gbm_frame(X_va, use_shortcuts, numeric))[:, 1]
    models[name] = model
    results.append({**evaluate(y_val, scores, name), "trees": model.best_iteration_})


run_gbm("LightGBM, no shortcut columns", X_train, X_val, False)
run_gbm("LightGBM, with format and currency", X_train, X_val, True)

pd.DataFrame(results)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr,trees
0,ACH rule,0.006691,0.000000,0.000000,NaN
1,"LogReg, no shortcut columns",0.002210,0.001931,0.018340,NaN
2,"LogReg, with format and currency",0.016349,0.031853,0.131274,NaN
3,"LightGBM, no shortcut columns",0.006639,0.015444,0.122587,49.0
4,"LightGBM, with format and currency",0.075553,0.157336,0.500965,95.0


**Finding:** With the same features, LightGBM beats logistic regression. Without the shortcut columns, recall at a 1% false positive rate rises from 19 to 127 of 1,036 laundering transactions, which supports the view that the linear model could not represent the amount band seen in the EDA. PR-AUC without shortcut columns is still only 0.0066, level with the ACH rule (0.0067), so transaction level features remain the main limit. With payment format and payment currency, PR-AUC reaches 0.0756 and the model catches 519 of 1,036 laundering transactions (50%) at a 1% false positive rate, about one true alert in 20, against one in 131 for the ACH rule. The shortcut columns multiply PR-AUC by about eleven, so this model depends heavily on them.

## Graph derived features

Each transaction gets counts describing the recent activity of its sender and receiver. Every count uses only transactions with an earlier timestamp, excludes self transfers, and covers a trailing window of 1 or 3 days.

In [8]:
BIG = 100_000
WINDOWS = {"1d": 1440, "3d": 4320}  # minutes


def account_codes(tx):
    from_key = tx["from_bank"].astype(str) + "_" + tx["from_account"].astype(str)
    to_key = tx["to_bank"].astype(str) + "_" + tx["to_account"].astype(str)
    codes, _ = pd.factorize(pd.concat([from_key, to_key], ignore_index=True))
    return codes[: len(tx)], codes[len(tx):]


def count_prior(event_codes, event_minutes, query_codes, query_minutes, window):
    events = np.sort(event_codes.astype(np.int64) * BIG + event_minutes)
    q = query_codes.astype(np.int64) * BIG + query_minutes
    return np.searchsorted(events, q, "left") - np.searchsorted(events, q - window, "left")


def add_graph_features(tx, windows=WINDOWS):
    from_code, to_code = account_codes(tx)
    minutes = (
        (tx["timestamp"] - tx["timestamp"].min()).dt.total_seconds() // 60
    ).to_numpy().astype(np.int64)

    real = ~tx["self_transfer"].to_numpy()
    ev_from, ev_to, ev_min = from_code[real], to_code[real], minutes[real]

    out = pd.DataFrame(index=tx.index)
    for label, w in windows.items():
        out[f"sender_out_{label}"] = count_prior(ev_from, ev_min, from_code, minutes, w)
        out[f"sender_in_{label}"] = count_prior(ev_to, ev_min, from_code, minutes, w)
        out[f"receiver_in_{label}"] = count_prior(ev_to, ev_min, to_code, minutes, w)
        out[f"receiver_out_{label}"] = count_prior(ev_from, ev_min, to_code, minutes, w)

    n_codes = int(max(from_code.max(), to_code.max())) + 1
    _, pair_code = np.unique(from_code.astype(np.int64) * n_codes + to_code, return_inverse=True)
    out["pair_prior_3d"] = count_prior(pair_code[real], ev_min, pair_code, minutes, WINDOWS["3d"])
    return out


G = add_graph_features(tx)
GRAPH = list(G.columns)
G.describe().T[["mean", "50%", "max"]]

,mean,50%,max
sender_out_1d,849.619720,3.0,26368.0
sender_in_1d,6.574134,1.0,539.0
receiver_in_1d,2.407124,2.0,538.0
receiver_out_1d,8.385593,0.0,26359.0
sender_out_3d,2052.326746,8.0,59868.0
sender_in_3d,12.369016,3.0,545.0
receiver_in_3d,5.766074,5.0,544.0
receiver_out_3d,18.100522,0.0,59867.0
pair_prior_3d,3.554806,3.0,24.0


In [9]:
check = tx.join(G)
fan = check[
    (check["from_account"] == "800737690") & (check["is_laundering"] == 1)
].sort_values("timestamp")
fan[["timestamp", "to_account", "sender_out_1d", "sender_out_3d", "is_laundering"]]

,timestamp,to_account,sender_out_1d,sender_out_3d,is_laundering
137520,2022-09-01 00:06:00,80011F990,0,0,1
137522,2022-09-01 04:33:00,80020C5B0,1,1,1
137524,2022-09-01 09:14:00,80006A5E0,2,2,1
137525,2022-09-01 09:56:00,8007A5B70,3,3,1
137526,2022-09-01 11:28:00,80093C0D0,4,4,1
137527,2022-09-01 13:13:00,80078E200,5,5,1
137529,2022-09-01 14:11:00,80066B990,6,6,1
137531,2022-09-02 15:40:00,8002CC310,0,7,1
137532,2022-09-02 21:23:00,8004030A0,1,8,1
137533,2022-09-02 23:10:00,800578800,2,9,1


## LightGBM with graph derived features

Same model and settings as before, with the nine graph derived features added.

In [10]:
ALL_NUMERIC = NUMERIC + GRAPH

X_train_g = X_train.join(G)
X_val_g = X_val.join(G)

run_gbm("LightGBM + graph, no shortcut columns", X_train_g, X_val_g, False, ALL_NUMERIC)
run_gbm("LightGBM + graph, with format and currency", X_train_g, X_val_g, True, ALL_NUMERIC)

pd.DataFrame(results)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr,trees
0,ACH rule,0.006691,0.000000,0.000000,NaN
1,"LogReg, no shortcut columns",0.002210,0.001931,0.018340,NaN
2,"LogReg, with format and currency",0.016349,0.031853,0.131274,NaN
3,"LightGBM, no shortcut columns",0.006639,0.015444,0.122587,49.0
4,"LightGBM, with format and currency",0.075553,0.157336,0.500965,95.0
5,"LightGBM + graph, no shortcut columns",0.270240,0.323359,0.662162,118.0
6,"LightGBM + graph, with format and currency",0.446412,0.508687,0.770270,195.0


**Finding:** Adding nine graph derived features changes the picture. Without the shortcut columns, LightGBM reaches a PR-AUC of 0.2702, about forty times the same model without graph features (0.0066) and more than three times the earlier model that used payment format and currency (0.0756). It catches 686 of 1,036 laundering transactions (66%) at a 1% false positive rate, about one true alert in 15, and 335 (32%) at a 0.1% false positive rate, about one in 4. With the shortcut columns PR-AUC rises to 0.4464 and recall at a 1% false positive rate to 798 (77%), so they now multiply PR-AUC by about 1.7, against about eleven before. Most of the signal is in how accounts behave across transactions. Every count uses only earlier timestamps, which was verified by hand on a known fan-out account. Validation scores are slightly optimistic because early stopping uses the validation set. The test set gives the unbiased figure.

## Stability across random seeds

The graph feature models are retrained with three other seeds. The seed controls which rows and features each tree samples, so a stable model should score similarly under each.

In [11]:
rows = []
for seed in [0, 1, 2]:
    for label, use_shortcuts in [("no shortcuts", False), ("with shortcuts", True)]:
        model = fit_gbm(X_train_g, y_train, X_val_g, y_val, use_shortcuts, ALL_NUMERIC, seed=seed)
        scores = model.predict_proba(gbm_frame(X_val_g, use_shortcuts, ALL_NUMERIC))[:, 1]
        metrics = evaluate(y_val, scores, label)
        rows.append({
            "features": label,
            "seed": seed,
            "pr_auc": metrics["pr_auc"],
            "recall_at_1.0%_fpr": metrics["recall_at_1.0%_fpr"],
            "trees": model.best_iteration_,
        })

seed_runs = pd.DataFrame(rows).sort_values(["features", "seed"])
display(seed_runs)
seed_runs.groupby("features")["pr_auc"].agg(["min", "mean", "max"]).round(4)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_s

,features,seed,pr_auc,recall_at_1.0%_fpr,trees
0,no shortcuts,0,0.282166,0.662162,104
2,no shortcuts,1,0.269629,0.669884,107
4,no shortcuts,2,0.273508,0.670849,128
1,with shortcuts,0,0.446111,0.770270,176
3,with shortcuts,1,0.446709,0.769305,183
5,with shortcuts,2,0.443732,0.771236,176


,min,mean,max
features,,,
no shortcuts,0.2696,0.2751,0.2822
with shortcuts,0.4437,0.4455,0.4467


**Finding:** Results are stable across seeds. Without the shortcut columns, PR-AUC ranges from 0.2696 to 0.2822 over three further seeds (mean 0.2751), with recall at a 1% false positive rate between 66% and 67% and 104 to 128 trees. With the shortcut columns, PR-AUC ranges from 0.4437 to 0.4467 (mean 0.4455), with recall at a 1% false positive rate of 77% and 176 to 183 trees. The main results, trained with seed 42, fall inside both ranges. This shows the scores do not depend on the random sampling during training. It does not show how the model performs on other days of data, which the test set addresses.

## Permutation importance

Each feature is shuffled in the validation set, three times, and the average drop in PR-AUC is recorded. A large drop means the model relies on that feature. This replaces an earlier gain based table, which was distorted by the unstable training run.

In [12]:
def permutation_importance(model, frame, y, n_repeats=3, seed=0):
    rng = np.random.default_rng(seed)
    base = average_precision_score(y, model.predict_proba(frame)[:, 1])
    work = frame.copy()
    rows = []
    for col in frame.columns:
        original = work[col].copy()
        drops = []
        for _ in range(n_repeats):
            work[col] = frame[col].iloc[rng.permutation(len(frame))].values
            drops.append(base - average_precision_score(y, model.predict_proba(work)[:, 1]))
        work[col] = original
        rows.append({
            "feature": col,
            "pr_auc_drop": np.mean(drops),
            "share_of_base": np.mean(drops) / base,
        })
    table = pd.DataFrame(rows).set_index("feature").sort_values("pr_auc_drop", ascending=False)
    return base, table


for name, use_shortcuts in [
    ("LightGBM + graph, no shortcut columns", False),
    ("LightGBM + graph, with format and currency", True),
]:
    frame = gbm_frame(X_val_g, use_shortcuts, ALL_NUMERIC)
    base, table = permutation_importance(models[name], frame, y_val)
    print(f"{name} | baseline PR-AUC: {base:.4f}")
    display(table.round(4))

LightGBM + graph, no shortcut columns | baseline PR-AUC: 0.2702


,pr_auc_drop,share_of_base
feature,,
pair_prior_3d,0.2678,0.9908
amount_vs_currency,0.2162,0.7999
self_transfer,0.1940,0.7177
receiver_in_1d,0.1338,0.4952
log_amount,0.0755,0.2793
sender_in_3d,0.0710,0.2627
receiver_in_3d,0.0694,0.2567
receiver_out_3d,0.0365,0.1352
hour_cos,0.0157,0.0580


LightGBM + graph, with format and currency | baseline PR-AUC: 0.4464


,pr_auc_drop,share_of_base
feature,,
pair_prior_3d,0.4320,0.9676
payment_format,0.3976,0.8906
self_transfer,0.3439,0.7705
amount_vs_currency,0.3250,0.7281
receiver_in_1d,0.1471,0.3296
sender_in_3d,0.1084,0.2427
receiver_in_3d,0.1045,0.2340
receiver_out_3d,0.0521,0.1168
payment_currency,0.0245,0.0549


In [13]:
v = val.join(G)
v["new_pair"] = (v["pair_prior_3d"] == 0) & (~v["self_transfer"])

print(
    v.groupby("new_pair")["is_laundering"]
    .agg(rows="size", laundering="sum", rate="mean")
)

rule = v["new_pair"].to_numpy().astype(int)
tp = int(((rule == 1) & (y_val == 1)).sum())
fp = int(((rule == 1) & (y_val == 0)).sum())
print(f"\nrule: flag every transfer to a new counterparty")
print(f"alerts: {tp + fp:,}   precision: {tp / (tp + fp):.2%}   recall: {tp / y_val.sum():.2%}")
print(evaluate(y_val, rule, "New counterparty rule"))

            rows  laundering      rate
new_pair                              
False     920963         194  0.000211
True       44561         842  0.018895

rule: flag every transfer to a new counterparty
alerts: 44,561   precision: 1.89%   recall: 81.27%
{'model': 'New counterparty rule', 'pr_auc': 0.015558037313458095, 'recall_at_0.1%_fpr': 0.0, 'recall_at_1.0%_fpr': 0.0}


**Finding:** One feature dominates. Shuffling `pair_prior_3d`, the number of earlier transfers between the same sender and receiver in the previous 3 days, removes 99% of PR-AUC without the shortcut columns and 97% with them. The model has learned that laundering goes to counterparties the sender has not paid recently. `self_transfer` (72% drop) acts as a tie breaker, because self transfers also have a pair count of zero. Amount relative to the currency norm matters (80%) far more than raw log amount (28%), and receiver activity matters (`receiver_in_1d` 50%) while the sender's outgoing counts contribute nothing, against the expectation that fan-out would show up there. Where available, `payment_format` is heavily used (89%) and `payment_currency` hardly at all (5%), so payment format is the real shortcut. Hour of day has little effect.

A rule that flags every transfer to a new counterparty confirms this. Such transfers are 4.6% of validation rows but contain 842 of 1,036 laundering transactions, a rate of 1.89% against 0.02% elsewhere. The rule reaches 81.3% recall at 1.89% precision with 44,561 alerts, against 85.9% recall at 0.76% precision and 116,910 alerts for the ACH rule, and has a PR-AUC of 0.0156. It replaces the ACH rule as the baseline to beat. The graph model without shortcut columns scores 0.2702, about 17 times higher, because it ranks these transfers using amount and receiver activity.

This signal is likely stronger here than in real data. Only 4.6% of transfers go to a counterparty not paid in the previous 3 days, which is far more repetitive than real customer behaviour. It uses only earlier transactions and no labels, so it is not leakage, but it is a third signal that depends on how the data was generated, alongside payment format and currency.

## Ablation: retraining without the pair count

Permutation importance scrambles a feature the model was trained to expect, which overstates dependence. Here the graph models are retrained without `pair_prior_3d`, to see how much the remaining features recover.

In [14]:
NO_PAIR = [c for c in ALL_NUMERIC if c != "pair_prior_3d"]

run_gbm("LightGBM + graph, no shortcuts, no pair count", X_train_g, X_val_g, False, NO_PAIR)
run_gbm("LightGBM + graph, with shortcuts, no pair count", X_train_g, X_val_g, True, NO_PAIR)

pd.DataFrame(results).tail(4)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr,trees
5,"LightGBM + graph, no shortcut columns",0.270240,0.323359,0.662162,118.0
6,"LightGBM + graph, with format and currency",0.446412,0.508687,0.770270,195.0
7,"LightGBM + graph, no shortcuts, no pair count",0.104354,0.158301,0.423745,60.0
8,"LightGBM + graph, with shortcuts, no pair count",0.258917,0.362934,0.636100,251.0


**Finding:** Retraining without `pair_prior_3d` shows partial redundancy. Without the shortcut columns, PR-AUC falls from 0.2702 to 0.1044, a 61% drop, against the 99% drop from permutation. The remaining graph features still give about sixteen times the PR-AUC of the model with no graph features (0.0066), and catch 439 of 1,036 laundering transactions (42%) at a 1% false positive rate. With the shortcut columns, PR-AUC falls from 0.4464 to 0.2589. The pair count therefore multiplies PR-AUC by about 2.6 without shortcuts and about 1.7 with them. It is the strongest single feature, but the result does not rest on it alone. Because the pair count likely benefits from the simulator's repetitive normal behaviour, the behaviour only model is best described by a range: 0.10 from account activity counts alone, and 0.27 with the new counterparty signal included.

## Error analysis by typology

Recall is measured separately for each laundering typology in the patterns file, and for laundering that belongs to no listed pattern. The threshold is the score that gives a 1% false positive rate on validation.

In [15]:
KEY = ["timestamp", "from_bank", "from_account", "to_bank", "to_account", "amount_paid"]


def parse_patterns(path):
    rows, attempt_id, typology = [], -1, None
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line.startswith("BEGIN LAUNDERING ATTEMPT"):
                attempt_id += 1
                typology = line.split(" - ", 1)[1].split(":")[0].strip()
            elif line and not line.startswith("END LAUNDERING ATTEMPT"):
                rows.append(line.split(",") + [attempt_id, typology])
    p = pd.DataFrame(rows, columns=TX_COLS + ["attempt_id", "typology"])
    p["timestamp"] = pd.to_datetime(p["timestamp"], format="%Y/%m/%d %H:%M")
    p["from_bank"] = p["from_bank"].astype("int32")
    p["to_bank"] = p["to_bank"].astype("int32")
    p["amount_paid"] = p["amount_paid"].astype(float)
    return p


patterns = parse_patterns(DATA / "HI-Small_Patterns.txt")
pattern_key = patterns.drop_duplicates(KEY)[KEY + ["attempt_id", "typology"]]


def with_typology(frame):
    out = frame[KEY].copy()
    out["from_account"] = out["from_account"].astype(str)
    out["to_account"] = out["to_account"].astype(str)
    merged = out.merge(pattern_key, on=KEY, how="left")
    assert len(merged) == len(frame)
    return merged["typology"].fillna("No listed pattern").to_numpy()


val_typology = with_typology(val)

check = pd.DataFrame({"typology": val_typology, "is_laundering": y_val})
print("pattern rows matched in validation:", (val_typology != "No listed pattern").sum())
print("matched rows that are not laundering:",
      ((check["typology"] != "No listed pattern") & (check["is_laundering"] == 0)).sum())
check[check["is_laundering"] == 1]["typology"].value_counts()

pattern rows matched in validation: 657
matched rows that are not laundering: 0


typology
No listed pattern    379
STACK                150
SCATTER-GATHER       134
GATHER-SCATTER        96
CYCLE                 66
FAN-OUT               64
FAN-IN                60
BIPARTITE             49
RANDOM                38
Name: count, dtype: int64

In [16]:
def recall_by_typology(scores, y, typology, fpr=0.01):
    threshold = np.quantile(scores[y == 0], 1 - fpr)
    table = pd.DataFrame({"typology": typology[y == 1], "caught": scores[y == 1] > threshold})
    return (
        table.groupby("typology")["caught"]
        .agg(laundering="size", caught="sum", recall="mean")
        .sort_values("recall", ascending=False)
    )


for name, use_shortcuts in [
    ("LightGBM + graph, no shortcut columns", False),
    ("LightGBM + graph, with format and currency", True),
]:
    scores = models[name].predict_proba(gbm_frame(X_val_g, use_shortcuts, ALL_NUMERIC))[:, 1]
    print(name)
    display(recall_by_typology(scores, y_val, val_typology).round(3))

LightGBM + graph, no shortcut columns


,laundering,caught,recall
typology,,,
FAN-IN,60,55,0.917
BIPARTITE,49,44,0.898
STACK,150,134,0.893
SCATTER-GATHER,134,117,0.873
GATHER-SCATTER,96,83,0.865
RANDOM,38,31,0.816
CYCLE,66,52,0.788
FAN-OUT,64,50,0.781
No listed pattern,379,120,0.317


LightGBM + graph, with format and currency


,laundering,caught,recall
typology,,,
CYCLE,66,66,1.000
RANDOM,38,38,1.000
FAN-IN,60,60,1.000
GATHER-SCATTER,96,95,0.990
FAN-OUT,64,63,0.984
STACK,150,147,0.980
SCATTER-GATHER,134,131,0.978
BIPARTITE,49,47,0.959
No listed pattern,379,151,0.398


In [17]:
scores = models["LightGBM + graph, no shortcut columns"].predict_proba(
    gbm_frame(X_val_g, False, ALL_NUMERIC)
)[:, 1]
print("Recall at a 0.1% false positive rate, no shortcut columns")
display(recall_by_typology(scores, y_val, val_typology, fpr=0.001).round(3))

v = val.join(G)
v["group"] = np.where(val_typology == "No listed pattern", "unpatterned", "patterned")
v["new_pair"] = (v["pair_prior_3d"] == 0) & (~v["self_transfer"])
v["is_ach"] = v["payment_format"] == "ACH"

print("Profile of laundering transactions in validation")
display(
    v[v["is_laundering"] == 1]
    .groupby("group")
    .agg(
        rows=("is_ach", "size"),
        share_ach=("is_ach", "mean"),
        share_new_pair=("new_pair", "mean"),
        median_amount=("amount_paid", "median"),
        median_receiver_in_1d=("receiver_in_1d", "median"),
    )
    .round(3)
)

Recall at a 0.1% false positive rate, no shortcut columns


,laundering,caught,recall
typology,,,
FAN-IN,60,39,0.650
GATHER-SCATTER,96,60,0.625
BIPARTITE,49,25,0.510
SCATTER-GATHER,134,66,0.493
CYCLE,66,29,0.439
STACK,150,65,0.433
FAN-OUT,64,27,0.422
RANDOM,38,13,0.342
No listed pattern,379,11,0.029


Profile of laundering transactions in validation


,rows,share_ach,share_new_pair,median_amount,median_receiver_in_1d
group,,,,,
patterned,657,1.000,0.941,12260.49,2.0
unpatterned,379,0.615,0.591,2757.80,2.0


**Finding:** The overall recall hides a sharp split. At a 1% false positive rate, the model without shortcut columns catches 566 of 657 laundering transactions that belong to a listed pattern (86%), but only 120 of 379 that belong to none (32%). Every one of the eight typologies is caught at least 78% of the time, from 78% for fan-out to 92% for fan-in. With the shortcut columns, recall on patterned laundering reaches 98%, which reflects that all pattern transactions in validation use ACH.

At a 0.1% false positive rate the typologies separate. Recall on patterned laundering falls to 49% (324 of 657), ranging from 34% for random chains to 65% for fan-in, and unpatterned laundering is almost entirely missed (11 of 379). The two best typologies, fan-in and gather-scatter, both involve many accounts paying one receiver, which matches the importance of receiver activity. Typology groups hold 38 to 150 transactions, so differences of a few points are within noise.

Unpatterned laundering is missed because it resembles normal activity. Only 59% of it goes to a new counterparty, against 94% for patterned laundering, only 62% uses ACH, and its median amount is 2,758 against 12,260.

For the planned graph neural network, these results set the target. At a 1% false positive rate there is little room to improve on listed patterns. At 0.1% there is substantial room, especially on multi-hop typologies such as cycles (44%), stacks (43%) and random chains (34%). Whether a graph model helps on unpatterned laundering is an open question.

## Diagnostic: what is unpatterned laundering?

The model misses most laundering outside the listed patterns. This compares normal, patterned and unpatterned transfers on their links to earlier laundering and on pass-through behaviour. The link columns use true labels of earlier transactions, so they are a diagnostic and not usable features.

In [18]:
from_code, to_code = account_codes(tx)
minutes = (
    (tx["timestamp"] - tx["timestamp"].min()).dt.total_seconds() // 60
).to_numpy().astype(np.int64)
real = ~tx["self_transfer"].to_numpy()
laund = (tx["is_laundering"].to_numpy() == 1) & real


def minutes_since_last(event_codes, event_minutes, query_codes, query_minutes):
    events = np.sort(event_codes.astype(np.int64) * BIG + event_minutes)
    q = query_codes.astype(np.int64) * BIG + query_minutes
    pos = np.searchsorted(events, q, "left") - 1
    prev = events[np.clip(pos, 0, None)]
    same_account = (pos >= 0) & (prev // BIG == query_codes)
    return np.where(same_account, q - prev, np.nan)


D = pd.DataFrame(index=tx.index)
D["sender_received_laundering_3d"] = count_prior(to_code[laund], minutes[laund], from_code, minutes, 4320) > 0
D["sender_sent_laundering_3d"] = count_prior(from_code[laund], minutes[laund], from_code, minutes, 4320) > 0
D["receiver_received_laundering_3d"] = count_prior(to_code[laund], minutes[laund], to_code, minutes, 4320) > 0
D["mins_since_sender_received"] = minutes_since_last(to_code[real], minutes[real], from_code, minutes)

d = val.join(D).join(G[["sender_in_1d"]])
d["group"] = np.where(
    y_val == 0,
    "normal",
    np.where(val_typology == "No listed pattern", "unpatterned laundering", "patterned laundering"),
)
d["sender_received_money_1d"] = d["sender_in_1d"] > 0
d = d[~d["self_transfer"]]

d.groupby("group").agg(
    rows=("group", "size"),
    sender_received_laundering_3d=("sender_received_laundering_3d", "mean"),
    sender_sent_laundering_3d=("sender_sent_laundering_3d", "mean"),
    receiver_received_laundering_3d=("receiver_received_laundering_3d", "mean"),
    sender_received_money_1d=("sender_received_money_1d", "mean"),
    median_mins_since_sender_received=("mins_since_sender_received", "median"),
).round(3).T

group,normal,patterned laundering,unpatterned laundering
rows,943690.000,657.000,379.000
sender_received_laundering_3d,0.004,0.458,0.005
sender_sent_laundering_3d,0.109,0.335,0.364
receiver_received_laundering_3d,0.004,0.289,0.016
sender_received_money_1d,0.566,0.619,0.237
median_mins_since_sender_received,995.000,790.000,6540.000


In [19]:
cols = ["sender_out_3d", "sender_in_3d", "receiver_in_3d", "receiver_out_3d", "pair_prior_3d"]
d2 = d.join(G[cols])

print("Median recent activity by group")
display(d2.groupby("group")[cols].median().T)

print("Sender outgoing transfers in the last 3 days, by group")
display(d2.groupby("group")["sender_out_3d"].quantile([0.25, 0.5, 0.75, 0.9]).unstack().round(0))

unp = d2[d2["group"] == "unpatterned laundering"]
sender_id = unp["from_bank"].astype(str) + "_" + unp["from_account"].astype(str)
counts = sender_id.value_counts()
print("unpatterned laundering transfers:", len(unp))
print("distinct sending accounts:", counts.size)
print("transfers from the 10 busiest of those senders:", int(counts.head(10).sum()))

Median recent activity by group


group,normal,patterned laundering,unpatterned laundering
sender_out_3d,15.0,3.0,4.0
sender_in_3d,4.0,6.0,0.0
receiver_in_3d,7.0,5.0,5.0
receiver_out_3d,0.0,1.0,0.0
pair_prior_3d,6.0,0.0,0.0


Sender outgoing transfers in the last 3 days, by group


,0.25,0.50,0.75,0.90
group,,,,
normal,8.0,15.0,28.0,3073.0
patterned laundering,0.0,3.0,11.0,22.0
unpatterned laundering,0.0,4.0,5086.0,51136.0


unpatterned laundering transfers: 379
distinct sending accounts: 259
transfers from the 10 busiest of those senders: 122


In [20]:
scores = models["LightGBM + graph, no shortcut columns"].predict_proba(
    gbm_frame(X_val_g, False, ALL_NUMERIC)
)[:, 1]
threshold = np.quantile(scores[y_val == 0], 0.99)
caught = pd.Series(scores > threshold, index=val.index)

u = d2[d2["group"] == "unpatterned laundering"].copy()
u["sender_type"] = np.where(u["sender_out_3d"] >= 1000, "hub sender", "quiet sender")
u["caught"] = caught.loc[u.index]
u["new_pair"] = u["pair_prior_3d"] == 0

display(
    u.groupby("sender_type").agg(
        rows=("caught", "size"),
        caught=("caught", "sum"),
        recall=("caught", "mean"),
        share_new_pair=("new_pair", "mean"),
        median_amount=("amount_paid", "median"),
        median_sender_in_3d=("sender_in_3d", "median"),
    ).round(3)
)

normal = d2[d2["group"] == "normal"]
print(f"share of normal transfers sent by hub accounts: {(normal['sender_out_3d'] >= 1000).mean():.1%}")

,rows,caught,recall,share_new_pair,median_amount,median_sender_in_3d
sender_type,,,,,,
hub sender,133,0,0.000,0.000,1320.14,0.0
quiet sender,246,120,0.488,0.911,3120.98,0.0


share of normal transfers sent by hub accounts: 10.5%


**Finding:** Unpatterned laundering is two different groups. Neither is downstream of other laundering: only 0.5% of its senders received laundering money in the previous 3 days, the same as normal senders (0.4%). Neither is pass through: only 24% of its senders received any money in the previous 24 hours, against 57% of normal senders. The 379 transfers come from 259 accounts, with the ten busiest sending 122.

About a third (133 transfers) is sent by hub accounts with 1,000 or more outgoing transfers in 3 days. The model catches none of these at a 1% false positive rate. Every one goes to a counterparty the hub already paid, at a median amount of 1,320, close to the normal median. Hubs send 10.5% of normal transfers, so the laundering rate among hub transfers is about 0.13%, barely above the overall rate. On the current features this group is not distinguishable from the hub's normal activity.

The other two thirds (246 transfers) are sent by quiet accounts. The model catches 120 of them (49%). They resemble patterned laundering: 91% go to a new counterparty and the median amount is 3,121.

Of the 350 laundering transfers missed at a 1% false positive rate, 91 are patterned, 126 are unpatterned from quiet senders, and 133 are unpatterned from hubs. Hub sent laundering is 13% of all laundering in validation, which caps recall at about 87% unless it can be detected. From here, recall is reported separately for patterned, quiet sender and hub sender laundering.

## Account attributes and counterparty breadth

Two feature sets aimed at laundering sent by quiet accounts, which have little transaction history. Account attributes describe the owner and bank of each side. Counterparty breadth counts recent transfers to or from new counterparties. Account attributes may reflect how the simulator assigns laundering, so models are reported with and without them.

In [21]:
accounts = pd.read_csv(DATA / "HI-Small_accounts.csv")
accounts.columns = ["bank_name", "bank_id", "account", "entity_id", "entity_name"]
accounts["entity_type"] = accounts["entity_name"].str.replace(r"\s*#\d+$", "", regex=True)
accounts["bank_country"] = accounts["bank_name"].str.extract(r"^(.+?) Bank #\d+$")[0].fillna("Other")
accounts["entity_accounts"] = accounts.groupby("entity_id")["account"].transform("size")
accounts = accounts.drop_duplicates(["bank_id", "account"])

n_codes = int(max(from_code.max(), to_code.max())) + 1
seen = pd.concat([
    pd.DataFrame({"code": from_code, "bank_id": tx["from_bank"].to_numpy(),
                  "account": tx["from_account"].astype(str).to_numpy()}).drop_duplicates("code"),
    pd.DataFrame({"code": to_code, "bank_id": tx["to_bank"].to_numpy(),
                  "account": tx["to_account"].astype(str).to_numpy()}).drop_duplicates("code"),
]).drop_duplicates("code")
by_code = (
    seen.merge(accounts, on=["bank_id", "account"], how="left")
    .set_index("code")
    .reindex(range(n_codes))
)
print(f"accounts matched to attributes: {by_code['entity_id'].notna().mean():.2%}")

A = pd.DataFrame(index=tx.index)
for side, code in [("sender", from_code), ("receiver", to_code)]:
    A[f"{side}_entity_type"] = by_code["entity_type"].to_numpy()[code]
    A[f"{side}_bank_country"] = by_code["bank_country"].to_numpy()[code]
    A[f"{side}_entity_accounts"] = by_code["entity_accounts"].to_numpy()[code]
entity = by_code["entity_id"].to_numpy()
A["same_entity"] = (entity[from_code] == entity[to_code]).astype(int)
A["same_bank"] = (tx["from_bank"].to_numpy() == tx["to_bank"].to_numpy()).astype(int)

first_contact = real & (G["pair_prior_3d"].to_numpy() == 0)
B = pd.DataFrame(index=tx.index)
B["sender_new_pair_out_3d"] = count_prior(
    from_code[first_contact], minutes[first_contact], from_code, minutes, WINDOWS["3d"])
B["receiver_new_pair_in_3d"] = count_prior(
    to_code[first_contact], minutes[first_contact], to_code, minutes, WINDOWS["3d"])

ATTRIBUTES = list(A.columns)
BREADTH = list(B.columns)
CATEGORIES.update({
    c: sorted(A[c].dropna().unique())
    for c in ATTRIBUTES if c.endswith(("entity_type", "bank_country"))
})

display(A.head(3).T)
B.describe().T[["mean", "50%", "max"]]

accounts matched to attributes: 100.00%


,0,1,2
sender_entity_type,Partnership,Partnership,Partnership
sender_bank_country,Other,Other,Other
sender_entity_accounts,7,6,7
receiver_entity_type,Partnership,Corporation,Partnership
receiver_bank_country,Other,Other,Other
receiver_entity_accounts,7,6,7
same_entity,1,0,1
same_bank,1,0,1


,mean,50%,max
sender_new_pair_out_3d,251.289568,1.0,14635.0
receiver_new_pair_in_3d,1.093613,1.0,544.0


In [22]:
X_train_x = X_train_g.join(B).join(A)
X_val_x = X_val_g.join(B).join(A)

run_gbm("LightGBM + graph + breadth, no shortcut columns",
        X_train_x, X_val_x, False, ALL_NUMERIC + BREADTH)
run_gbm("LightGBM + graph + breadth + attributes, no shortcut columns",
        X_train_x, X_val_x, False, ALL_NUMERIC + BREADTH + ATTRIBUTES)

pd.DataFrame(results).tail(5)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr,trees
6,"LightGBM + graph, with format and currency",0.446412,0.508687,0.770270,195.0
7,"LightGBM + graph, no shortcuts, no pair count",0.104354,0.158301,0.423745,60.0
8,"LightGBM + graph, with shortcuts, no pair count",0.258917,0.362934,0.636100,251.0
9,"LightGBM + graph + breadth, no shortcut columns",0.301810,0.353282,0.673745,193.0
10,"LightGBM + graph + breadth + attributes, no sh...",0.449695,0.482625,0.797297,828.0


In [23]:
val_group = np.where(
    val_typology != "No listed pattern",
    "patterned",
    np.where(G.loc[val.index, "sender_out_3d"].to_numpy() >= 1000,
             "unpatterned, hub sender", "unpatterned, quiet sender"),
)
GROUPS = ["patterned", "unpatterned, quiet sender", "unpatterned, hub sender"]


def recall_by_group(name, X_va, use_shortcuts, numeric):
    scores = models[name].predict_proba(gbm_frame(X_va, use_shortcuts, numeric))[:, 1]
    row = {"pr_auc": average_precision_score(y_val, scores)}
    for fpr in (0.01, 0.001):
        threshold = np.quantile(scores[y_val == 0], 1 - fpr)
        for group in GROUPS:
            mask = (y_val == 1) & (val_group == group)
            row[f"{group} @ {fpr:.1%}"] = (scores[mask] > threshold).mean()
    return pd.Series(row, name=name)


comparison = pd.DataFrame([
    recall_by_group("LightGBM + graph, no shortcut columns", X_val_g, False, ALL_NUMERIC),
    recall_by_group("LightGBM + graph + breadth, no shortcut columns",
                    X_val_x, False, ALL_NUMERIC + BREADTH),
    recall_by_group("LightGBM + graph + breadth + attributes, no shortcut columns",
                    X_val_x, False, ALL_NUMERIC + BREADTH + ATTRIBUTES),
])
comparison.round(3).T

,"LightGBM + graph, no shortcut columns","LightGBM + graph + breadth, no shortcut columns","LightGBM + graph + breadth + attributes, no shortcut columns"
pr_auc,0.270,0.302,0.450
patterned @ 1.0%,0.861,0.883,0.963
"unpatterned, quiet sender @ 1.0%",0.488,0.480,0.776
"unpatterned, hub sender @ 1.0%",0.000,0.000,0.015
patterned @ 0.1%,0.493,0.542,0.689
"unpatterned, quiet sender @ 0.1%",0.045,0.041,0.191
"unpatterned, hub sender @ 0.1%",0.000,0.000,0.000


In [24]:
name = "LightGBM + graph + breadth + attributes, no shortcut columns"
frame = gbm_frame(X_val_x, False, ALL_NUMERIC + BREADTH + ATTRIBUTES)
base, table = permutation_importance(models[name], frame, y_val, n_repeats=1)
print(f"baseline PR-AUC: {base:.4f}")
display(table.head(12).round(4))

t = train.join(A)
for col in ["sender_entity_type", "sender_bank_country", "receiver_bank_country", "same_bank", "same_entity"]:
    print(col)
    display(
        t.groupby(col)["is_laundering"]
        .agg(rows="size", laundering="sum", rate="mean")
        .sort_values("rate", ascending=False)
        .head(6)
    )

print("sender_entity_accounts")
bins = pd.cut(t["sender_entity_accounts"], [0, 1, 2, 5, 10, 50, 10_000])
display(t.groupby(bins, observed=True)["is_laundering"].agg(rows="size", laundering="sum", rate="mean"))

baseline PR-AUC: 0.4497


,pr_auc_drop,share_of_base
feature,,
pair_prior_3d,0.4267,0.9488
amount_vs_currency,0.2491,0.5538
sender_bank_country,0.2481,0.5516
receiver_bank_country,0.1837,0.4085
receiver_in_1d,0.1658,0.3686
sender_new_pair_out_3d,0.1430,0.3179
sender_entity_accounts,0.0927,0.2062
sender_in_3d,0.0738,0.1640
receiver_out_3d,0.0489,0.1088


sender_entity_type


,rows,laundering,rate
sender_entity_type,,,
Individual,2905,4,0.001377
Corporation,912191,917,0.001005
Partnership,1141210,847,0.000742
Sole Proprietorship,1134311,741,0.000653
Country,57974,21,0.000362
Direct,330,0,0.000000


sender_bank_country


,rows,laundering,rate
sender_bank_country,,,
Saudi Arabia,40109,117,0.002917
Finland,22129,34,0.001536
Portugal,20133,21,0.001043
Austria,24985,24,0.000961
Slovakia,8430,8,0.000949
Spain,72271,67,0.000927


... (4 outputs left out)

same_entity


,rows,laundering,rate
same_entity,,,
0,2653640,2509,0.000945
1,595281,21,0.000035


sender_entity_accounts


,rows,laundering,rate
sender_entity_accounts,,,
"(0, 1]",1174082,917,0.000781
"(1, 2]",214352,168,0.000784
"(2, 5]",559439,408,0.000729
"(5, 10]",539023,490,0.000909
"(10, 50]",196261,130,0.000662
"(50, 10000]",565764,417,0.000737


In [25]:
JURISDICTION = ["sender_bank_country", "receiver_bank_country"]
NO_JURISDICTION = [c for c in ATTRIBUTES if c not in JURISDICTION]

run_gbm("LightGBM + graph + breadth + attributes except bank country, no shortcut columns",
        X_train_x, X_val_x, False, ALL_NUMERIC + BREADTH + NO_JURISDICTION)

comparison = pd.DataFrame([
    recall_by_group("LightGBM + graph, no shortcut columns", X_val_g, False, ALL_NUMERIC),
    recall_by_group("LightGBM + graph + breadth, no shortcut columns",
                    X_val_x, False, ALL_NUMERIC + BREADTH),
    recall_by_group("LightGBM + graph + breadth + attributes except bank country, no shortcut columns",
                    X_val_x, False, ALL_NUMERIC + BREADTH + NO_JURISDICTION),
    recall_by_group("LightGBM + graph + breadth + attributes, no shortcut columns",
                    X_val_x, False, ALL_NUMERIC + BREADTH + ATTRIBUTES),
])
display(comparison.round(3).T)

t = train.join(A)
t = t[~t["self_transfer"]]
t["route"] = np.where(t["sender_bank_country"] == t["receiver_bank_country"],
                      "same country", "different country")
t.groupby("route")["is_laundering"].agg(rows="size", laundering="sum", rate="mean")

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,"LightGBM + graph, no shortcut columns","LightGBM + graph + breadth, no shortcut columns","LightGBM + graph + breadth + attributes except bank country, no shortcut columns","LightGBM + graph + breadth + attributes, no shortcut columns"
pr_auc,0.270,0.302,0.341,0.450
patterned @ 1.0%,0.861,0.883,0.913,0.963
"unpatterned, quiet sender @ 1.0%",0.488,0.480,0.764,0.776
"unpatterned, hub sender @ 1.0%",0.000,0.000,0.023,0.015
patterned @ 0.1%,0.493,0.542,0.531,0.689
"unpatterned, quiet sender @ 0.1%",0.045,0.041,0.179,0.191
"unpatterned, hub sender @ 0.1%",0.000,0.000,0.000,0.000


,rows,laundering,rate
route,,,
different country,856343,1423,0.001662
same country,1840192,1101,0.000598


**Finding:** The two feature sets do different jobs. Counterparty breadth raises PR-AUC from 0.270 to 0.302 and helps on patterned laundering at the strict budget (49% to 54% recall at a 0.1% false positive rate), but does nothing for quiet senders.

Owner attributes (entity type, accounts per entity, and same owner and same bank flags) raise PR-AUC to 0.341 and lift recall on unpatterned laundering from quiet senders from 48% to 76% at a 1% false positive rate, and from 4% to 18% at 0.1%. This is the gain on the group these features were built for. A likely reason, not yet confirmed, is that the same owner flag lets the model discount harmless transfers between accounts of one entity, which otherwise look like payments to a new counterparty.

Bank country adds a further large gain, to a PR-AUC of 0.450, mostly on patterned laundering at the strict budget (53% to 69%). Permutation importance puts sender and receiver bank country at 55% and 41%, yet no single country has an extreme laundering rate: the highest, Saudi Arabia, is about four times the average. The model uses country in combination with other features. Transfers between banks in different countries have a laundering rate 2.8 times that of transfers within a country, and bank country also partly restores the payment currency signal that was removed as a shortcut. Jurisdiction is a recognised risk factor, but the specific countries learned here belong to the simulator, so bank country is reported separately, like payment format.

Hub sent laundering is still undetected (at most 3 of 133), so the ceiling of about 87% recall remains.

The model without shortcut columns is therefore described by three levels: 0.302 from behaviour alone, 0.341 with owner attributes, and 0.450 with bank country. At a 1% false positive rate these catch about 67%, 76% and 80% of laundering.

## Stability of the new feature sets across seeds

The three models with breadth and attribute features are retrained with three other seeds. Recall on quiet sender laundering is tracked as well as PR-AUC, since that group holds only 246 transfers.

In [26]:
FEATURE_SETS = {
    "breadth": ALL_NUMERIC + BREADTH,
    "breadth + owner attributes": ALL_NUMERIC + BREADTH + NO_JURISDICTION,
    "breadth + all attributes": ALL_NUMERIC + BREADTH + ATTRIBUTES,
}
quiet = (y_val == 1) & (val_group == "unpatterned, quiet sender")

rows = []
for label, cols in FEATURE_SETS.items():
    for seed in [0, 1, 2]:
        model = fit_gbm(X_train_x, y_train, X_val_x, y_val, False, cols, seed=seed)
        scores = model.predict_proba(gbm_frame(X_val_x, False, cols))[:, 1]
        threshold = np.quantile(scores[y_val == 0], 0.99)
        rows.append({
            "features": label,
            "seed": seed,
            "pr_auc": average_precision_score(y_val, scores),
            "recall_at_1%_fpr": (scores[y_val == 1] > threshold).mean(),
            "quiet_recall_at_1%_fpr": (scores[quiet] > threshold).mean(),
            "trees": model.best_iteration_,
        })

seed_runs_new = pd.DataFrame(rows)
display(seed_runs_new.round(4))
(
    seed_runs_new.groupby("features", sort=False)[["pr_auc", "quiet_recall_at_1%_fpr"]]
    .agg(["min", "mean", "max"])
    .round(4)
)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_s

,features,seed,pr_auc,recall_at_1%_fpr,quiet_recall_at_1%_fpr,trees
0,breadth,0,0.3066,0.6670,0.4756,161
1,breadth,1,0.3022,0.6718,0.4919,180
2,breadth,2,0.2998,0.6660,0.4919,136
3,breadth + owner attributes,0,0.3419,0.7625,0.7642,1224
4,breadth + owner attributes,1,0.3383,0.7606,0.7683,959
5,breadth + owner attributes,2,0.3313,0.7597,0.7602,748
6,breadth + all attributes,0,0.4512,0.7944,0.7724,815
7,breadth + all attributes,1,0.4506,0.7857,0.7602,990
8,breadth + all attributes,2,0.4495,0.7867,0.7480,1082


pr_auc                 quiet_recall_at_1%_fpr  \
                               min    mean     max                    min   
features                                                                    
breadth                     0.2998  0.3029  0.3066                 0.4756   
breadth + owner attributes  0.3313  0.3371  0.3419                 0.7602   
breadth + all attributes    0.4495  0.4505  0.4512                 0.7480   

                                            
                              mean     max  
features                                    
breadth                     0.4864  0.4919  
breadth + owner attributes  0.7642  0.7683  
breadth + all attributes    0.7602  0.7724

**Finding:** All three models are stable across seeds. PR-AUC ranges from 0.300 to 0.307 with breadth, 0.331 to 0.342 with owner attributes, and 0.449 to 0.451 with all attributes, and the main results fall inside each range. Recall on unpatterned laundering from quiet senders at a 1% false positive rate is 48% to 49% with breadth and 76% to 77% with owner attributes, so the gain of about 28 points holds in every seed. The number of trees for the owner attributes model varies from 748 to 1,224 without a matching change in scores.

## Experiment: laundering sent by hub accounts

Hub sent laundering is 13% of laundering in validation and the models catch almost none of it. This tests whether comparing each transfer with the sender's own history helps. It first measures how well each feature separates laundering from normal transfers among hub transfers only, then trains a model on hub transfers alone.

In [27]:
from sklearn.metrics import roc_auc_score


def prior_mean(codes, minutes, values, include=None):
    if include is None:
        include = np.ones(len(values), dtype=bool)
    order = np.lexsort((minutes, codes))
    c = codes[order]
    v = np.where(include[order], values[order], 0.0)
    k = include[order].astype(np.int64)
    csum, ccnt = np.cumsum(v), np.cumsum(k)
    start = np.r_[True, c[1:] != c[:-1]]
    first = np.maximum.accumulate(np.where(start, np.arange(len(c)), 0))
    base_sum = np.where(first > 0, csum[first - 1], 0.0)
    base_cnt = np.where(first > 0, ccnt[first - 1], 0)
    prior_sum = csum - v - base_sum
    prior_cnt = ccnt - k - base_cnt
    out = np.full(len(values), np.nan)
    ok = prior_cnt > 0
    out[order[ok]] = prior_sum[ok] / prior_cnt[ok]
    return out


log_amount = np.log10(tx["amount_paid"].clip(lower=0.01).to_numpy())
_, pair_code = np.unique(from_code.astype(np.int64) * n_codes + to_code, return_inverse=True)

H = pd.DataFrame(index=tx.index)
H["amount_vs_pair"] = log_amount - prior_mean(pair_code, minutes, log_amount)
H["amount_vs_sender"] = log_amount - prior_mean(from_code, minutes, log_amount, include=real)
OWN_HISTORY = list(H.columns)

is_hub = pd.Series((G["sender_out_3d"].to_numpy() >= 1000) & real, index=tx.index)
hub_train = train.index[is_hub.loc[train.index].to_numpy()]
hub_val = val.index[is_hub.loc[val.index].to_numpy()]
yh_train = tx.loc[hub_train, "is_laundering"].to_numpy()
yh_val = tx.loc[hub_val, "is_laundering"].to_numpy()
print(f"hub transfers in train: {len(hub_train):,} with {yh_train.sum()} laundering")
print(f"hub transfers in validation: {len(hub_val):,} with {yh_val.sum()} laundering")

Xh_train = X_train_x.join(H).loc[hub_train]
Xh_val = X_val_x.join(H).loc[hub_val]

numeric_cols = [c for c in ALL_NUMERIC + BREADTH + ATTRIBUTES + OWN_HISTORY if c not in CATEGORIES]
separation = {}
for col in numeric_cols:
    values = Xh_val[col].fillna(Xh_val[col].median())
    if values.nunique() > 1:
        separation[col] = roc_auc_score(yh_val, values)
separation["is_ach (shortcut, for reference)"] = roc_auc_score(
    yh_val, (Xh_val["payment_format"] == "ACH").astype(int))

sep = pd.Series(separation, name="auc_within_hub_transfers")
sep.reindex((sep - 0.5).abs().sort_values(ascending=False).index).round(3).to_frame()

hub transfers in train: 255,396 with 364 laundering
hub transfers in validation: 99,452 with 133 laundering


,auc_within_hub_transfers
hour_sin,0.545
hour_cos,0.464
receiver_out_3d,0.528
sender_out_1d,0.481
sender_out_3d,0.481
receiver_out_1d,0.516
receiver_in_1d,0.513
amount_vs_currency,0.488
amount_vs_sender,0.489
amount_vs_pair,0.490


In [28]:
HUB_FEATURES = ALL_NUMERIC + BREADTH + ATTRIBUTES + OWN_HISTORY

rows = []
for label, use_shortcuts in [("no shortcut columns", False), ("with format and currency", True)]:
    model = fit_gbm(Xh_train, yh_train, Xh_val, yh_val, use_shortcuts, HUB_FEATURES)
    scores = model.predict_proba(gbm_frame(Xh_val, use_shortcuts, HUB_FEATURES))[:, 1]
    threshold = np.quantile(scores[yh_val == 0], 0.99)
    rows.append({
        "hub only model": label,
        "pr_auc": average_precision_score(yh_val, scores),
        "base_rate": yh_val.mean(),
        "roc_auc": roc_auc_score(yh_val, scores),
        "caught_at_1%_fpr": int((scores[yh_val == 1] > threshold).sum()),
        "laundering": int(yh_val.sum()),
        "trees": model.best_iteration_,
    })
pd.DataFrame(rows).round(4)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,hub only model,pr_auc,base_rate,roc_auc,caught_at_1%_fpr,laundering,trees
0,no shortcut columns,0.0015,0.0013,0.4899,2,133,9
1,with format and currency,0.0014,0.0013,0.4980,1,133,13


**Finding:** Hub sent laundering is not detectable with the available data. Among hub transfers in validation (99,452 transfers, 133 laundering), no feature separates laundering from normal activity: every single feature AUC lies between 0.46 and 0.55, including the two own history features (`amount_vs_pair` 0.490, `amount_vs_sender` 0.489) and the ACH flag (0.500). With 19 features tested on 133 positives, the most extreme value (0.545) is within what chance produces.

A model trained only on hub transfers, with 364 laundering examples, reaches a PR-AUC of 0.0015 against a base rate of 0.0013 and a ROC AUC of 0.49. It catches 2 of 133 at a 1% false positive rate, where chance would catch about 1, and early stopping ends training after 9 trees. Adding payment format and currency does not help.

The ceiling is therefore confirmed: about 13% of laundering in validation cannot be told apart from the same accounts' normal transfers using transaction behaviour, account attributes, or the account's own history, so recall cannot exceed about 87%. A likely explanation is that these transfers are labelled by the source of the funds, which is known to the simulator but not visible in the data. The own history features are not added to the main model.

In [29]:
final = pd.DataFrame(results)
final["pr_auc"] = final["pr_auc"].round(4)
final[["recall_at_0.1%_fpr", "recall_at_1.0%_fpr"]] = (
    final[["recall_at_0.1%_fpr", "recall_at_1.0%_fpr"]].round(3)
)
final

,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr,trees
0,ACH rule,0.0067,0.000,0.000,NaN
1,"LogReg, no shortcut columns",0.0022,0.002,0.018,NaN
2,"LogReg, with format and currency",0.0163,0.032,0.131,NaN
3,"LightGBM, no shortcut columns",0.0066,0.015,0.123,49.0
4,"LightGBM, with format and currency",0.0756,0.157,0.501,95.0
5,"LightGBM + graph, no shortcut columns",0.2702,0.323,0.662,118.0
6,"LightGBM + graph, with format and currency",0.4464,0.509,0.770,195.0
7,"LightGBM + graph, no shortcuts, no pair count",0.1044,0.158,0.424,60.0
8,"LightGBM + graph, with shortcuts, no pair count",0.2589,0.363,0.636,251.0
9,"LightGBM + graph + breadth, no shortcut columns",0.3018,0.353,0.674,193.0


## Summary

All results are on the validation set (7 to 8 Sep, 965,524 transfers, 1,036 laundering). The test set has not been used.

### Results ladder, without payment format or currency

| Step | PR-AUC | Recall at 1% false positive rate | Recall at 0.1% |
|---|---|---|---|
| Rule: flag every new counterparty | 0.016 | not usable | not usable |
| Logistic regression, transaction features | 0.002 | 2% | 0% |
| LightGBM, transaction features | 0.007 | 12% | 2% |
| + graph counts | 0.270 | 66% | 32% |
| + counterparty breadth | 0.302 | 67% | 35% |
| + owner attributes | 0.341 | 76% | 38% |
| + bank country | 0.450 | 80% | 48% |

### What was learned

1. **A single transaction carries almost no signal.** Transaction features alone score at the level of a one line rule. Counting each account's recent activity raises PR-AUC about forty times.
2. **Several strong signals depend on how the data was generated.** Payment format (ACH holds 87% of laundering), the count of earlier transfers between the same pair, and bank country. Each was measured by retraining without it. With payment format and currency added, the graph model scores 0.446. Without the pair count it scores 0.104.
3. **Training setup mattered.** Unweighted LightGBM with a fixed number of trees was unstable at this class balance. L2 regularisation and early stopping fixed it, and results are stable across seeds.
4. **Overall recall hides three groups.** For the model with owner attributes, at a 1% false positive rate: 91% of laundering in a listed pattern is caught, 76% of unpatterned laundering from quiet senders, and about 2% of unpatterned laundering from hub senders.
5. **Owner attributes are what help on quiet senders.** They raise recall on that group from 48% to 76%. Bank country adds little there and helps mainly on patterned laundering at the strict budget.

### Limits

- **A recall ceiling of about 87%.** Laundering sent by hub accounts is 13% of the total and could not be separated from those accounts' normal transfers by any feature or by a model trained on hub transfers alone.
- **Synthetic data.** Normal accounts here are far more repetitive than real customers, so the new counterparty signal is stronger than it would be on real data.
- **Validation scores are slightly optimistic,** because early stopping uses the validation set. The test set gives the unbiased figure.
- **Small groups.** Typology groups hold 38 to 150 laundering transfers, so differences of a few points between them are noise.

### Model carried forward

The main model is LightGBM with graph counts, counterparty breadth and owner attributes, without payment format, currency or bank country (PR-AUC 0.341, stable between 0.331 and 0.342 across seeds). It is the strongest model that does not lean on a signal tied to the simulator's payment or country assignments. The version with bank country (0.450) is reported alongside it as an upper variant.

### Next

The HuggingFace model is trained in its own notebook and compared with this one on validation. The better of the two is then scored once on the test set.